# Kalshi Sports markets and Yes-taker P&L

This notebook uses Kalshi's public API. It accepts positive-volume markets whose `result` is `yes` or `no`, regardless of settlement status. Single markets must belong to a Sports series, and every leg of a combo must be Sports-related. Zero-volume markets are discarded before combo occurrence lookups. The default configuration streams markets to CSV so the multi-million-row live universe is never held in memory.

In [1]:
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from kalshi_utils import (
    KalshiPublicClient,
    daily_market_summary,
    load_saved_markets,
    pull_sports_markets,
    pull_trade_summaries,
)

START_DATE = "2026-08-18"
END_DATE = None  # Yesterday in America/New_York
RUN_STARTED_AT = datetime.now(timezone.utc)
REFRESH_SINCE = None  # First run: None. Later: prior RUN_STARTED_AT.

LOW_MEMORY = True
MARKETS_CSV = Path("sports_markets.csv")
client = KalshiPublicClient(requests_per_second=10, timeout=45)

## 1. Pull Sports-related markets

Leave `LOW_MEMORY = True` for the initial backfill. API pages are filtered immediately, nested combo-leg payloads are omitted, and qualifying rows are flushed to CSV in 50,000-row batches.

In [3]:
if LOW_MEMORY:
    pull_sports_markets(
        START_DATE, END_DATE, client=client,
        archive_early_stop=True, refresh_since=REFRESH_SINCE,
        output_csv=MARKETS_CSV, return_dataframe=False,
        csv_flush_rows=500_000, include_combo_legs=False,
        qualification_log_every_pages=10, verbose=True,
    )
    markets = None
    print(f"Markets were streamed to {MARKETS_CSV.resolve()}")
else:
    markets = pull_sports_markets(
        START_DATE, END_DATE, client=client,
        archive_early_stop=True, refresh_since=REFRESH_SINCE,
        include_combo_legs=False,
        qualification_log_every_pages=10, verbose=True,
    )
    print(f"Markets retained in memory: {len(markets):,}")
    display(markets.head())

[00:40:01] Fetching Kalshi Sports series | elapsed 0.0s
[00:40:01] Sports series discovered: 3,478 | elapsed 0.4s
[00:40:01] Streaming markets from 2026-08-18T04:00:00+00:00 and filtering each page | elapsed 0.4s
[00:40:01] Historical markets: page 1, 0 potentially relevant rows streamed
[00:40:01] Historical scan reached settlements before 2026-08-18T04:00:00+00:00; stopping
[00:40:01] Historical market stream complete: 0 relevant rows
[00:40:02] Sports qualification page 1: received=1,000, qualified Sports=0 (single=0, combo=0) | elapsed 0.9s
[00:40:02] Live finalized: page 1, 1,000 rows streamed
[00:40:03] Sports qualification page 10: received=1,000, qualified Sports=0 (single=0, combo=0) | elapsed 2.3s
[00:40:03] Live finalized: page 10, 10,000 rows streamed
[00:40:05] Sports qualification page 20: received=1,000, qualified Sports=298 (single=0, combo=298) | elapsed 3.6s
[00:40:05] Live finalized: page 20, 20,000 rows streamed
[00:40:10] Sports qualification page 30: received=1,00

## 2. Daily totals without loading the full CSV

In low-memory mode this reads 100,000 rows at a time and retains only the small daily aggregation.

In [4]:
if markets is not None:
    daily_markets = daily_market_summary(markets)
else:
    daily_parts = []
    for chunk in pd.read_csv(MARKETS_CSV, chunksize=100_000):
        if chunk["is_combo"].dtype == object:
            chunk["is_combo"] = chunk["is_combo"].astype(str).str.lower().eq("true")
        else:
            chunk["is_combo"] = chunk["is_combo"].astype(bool)
        chunk["single_market"] = ~chunk["is_combo"]
        chunk["single_market_volume_contracts"] = chunk["market_volume_contracts"].where(~chunk["is_combo"], 0)
        chunk["combo_market_volume_contracts"] = chunk["market_volume_contracts"].where(chunk["is_combo"], 0)
        daily_parts.append(chunk.groupby("occurrence_date_ny", as_index=False).agg(
            market_count=("ticker", "size"),
            single_market_count=("single_market", "sum"),
            combo_market_count=("is_combo", "sum"),
            market_volume_contracts=("market_volume_contracts", "sum"),
            single_market_volume_contracts=("single_market_volume_contracts", "sum"),
            combo_market_volume_contracts=("combo_market_volume_contracts", "sum"),
        ))
    daily_markets = (pd.concat(daily_parts, ignore_index=True)
        .groupby("occurrence_date_ny", as_index=False).sum(numeric_only=True)
        .sort_values("occurrence_date_ny")) if daily_parts else pd.DataFrame()
daily_markets.tail(14)

,occurrence_date_ny,market_count,single_market_count,combo_market_count,market_volume_contracts
0,2026-08-18,262482,8388,254094,7.123444e+08
1,2026-08-19,374779,9487,365292,9.758720e+08
2,2026-08-20,280394,7802,272592,7.404378e+08
3,2026-08-21,218878,7763,211115,5.790291e+08
4,2026-08-22,489244,17113,472131,1.256783e+09


In [2]:
markets = load_saved_markets(
    "sports_markets.csv",
    "2026-08-18",
)

[01:49:35] Reading saved markets for 2026-08-18 through 2026-08-18 from sports_markets.csv | elapsed 0.0s
[01:49:36] Saved-market scan: 100,000 rows read, 0 matched | elapsed 0.7s
[01:49:43] Saved-market scan: 1,000,000 rows read, 15 matched | elapsed 7.5s
[01:49:49] Saved-market load complete: 262,482 rows returned | elapsed 13.8s


In [12]:
single_markets = markets[~markets["is_combo"]].copy()
combo_markets = markets[markets["is_combo"]].copy()

In [13]:
single_markets = single_markets.sort_values("market_volume_contracts", ascending=False).reset_index(drop=True)
combo_markets = combo_markets.sort_values("market_volume_contracts", ascending=False).reset_index(drop=True)

In [14]:
top_60_single_markets = single_markets.head(60).copy()
top_60_combo_markets = combo_markets.head(60).copy()

## 3. Pull trades for selected markets

Enter only the tickers you want. In low-memory mode the CSV is scanned in chunks and only matching rows are loaded. An empty list safely skips the trade pull.

In [15]:
MARKET_TICKERS = top_60_combo_markets["ticker"].tolist()

if markets is not None:
    selected_markets = markets[markets["ticker"].isin(MARKET_TICKERS)].copy()
else:
    selected_parts = []
    wanted = set(MARKET_TICKERS)
    if wanted:
        for chunk in pd.read_csv(MARKETS_CSV, chunksize=100_000):
            selected = chunk[chunk["ticker"].isin(wanted)]
            if not selected.empty:
                selected_parts.append(selected)
    selected_markets = pd.concat(selected_parts, ignore_index=True) if selected_parts else pd.DataFrame()

if selected_markets.empty:
    trade_summary = pd.DataFrame()
    print("No market tickers selected; trade pull skipped.")
else:
    trade_summary = pull_trade_summaries(
        selected_markets, client=client, max_workers=8, verbose=True,
    )
    display(trade_summary.head())

[01:58:28] Starting trade pull for 60 markets; 60 have nonzero API volume; 8 workers | elapsed 0.0s
[01:58:28] Trade summaries complete: 1/60 (2%) | elapsed 0.2s
[01:58:28] Trade summaries complete: 2/60 (3%) | elapsed 0.3s
[01:58:28] Trade summaries complete: 3/60 (5%) | elapsed 0.4s
[01:58:29] Trade summaries complete: 4/60 (7%) | elapsed 0.5s
[01:58:29] Trade summaries complete: 5/60 (8%) | elapsed 0.6s
[01:58:29] Trade summaries complete: 6/60 (10%) | elapsed 0.7s
[01:58:29] Trade summaries complete: 7/60 (12%) | elapsed 0.9s
[01:58:29] Trade summaries complete: 8/60 (13%) | elapsed 0.9s
[01:58:29] Trade summaries complete: 9/60 (15%) | elapsed 1.0s
[01:58:29] Trade summaries complete: 10/60 (17%) | elapsed 1.1s
[01:58:29] Trade summaries complete: 11/60 (18%) | elapsed 1.2s
[01:58:29] Trade summaries complete: 12/60 (20%) | elapsed 1.3s
[01:58:29] Trade summaries complete: 13/60 (22%) | elapsed 1.4s
[01:58:29] Trade summaries complete: 14/60 (23%) | elapsed 1.5s
[01:58:30] Trade s

,ticker,result,total_contract,yes_contract,yes_dollar_volume,yes_trade,no_contract,no_dollar_volume,no_trade,yes_taker_pnl
0,KXMVECROSSCATEGORY-SHARD1-S202601A0D883242-B5B...,no,623635.62,623635.62,5146.05413,978,0.00,0.00000,0,5146.05413
1,KXMVECROSSCATEGORY-SHARD1-S20260504D2385CB-E81...,no,782525.55,779193.76,2422.98171,224,3331.79,3253.24064,4,2422.98171
2,KXMVECROSSCATEGORY-SHARD1-S202608FACCCD5D2-F32...,no,252635.05,252635.05,2273.71545,1,0.00,0.00000,0,2273.71545
3,KXMVECROSSCATEGORY-SHARD1-S20260DA16EDA57E-050...,no,668397.65,668397.65,1124.77166,191,0.00,0.00000,0,1124.77166
4,KXMVECROSSCATEGORY-SHARD1-S20260E744575E70-2DF...,no,302727.27,302727.27,302.72727,1,0.00,0.00000,0,302.72727


`yes_taker_pnl` follows the requested market-maker counterparty convention: Yes dollar volume when the result is No; Yes dollar volume minus Yes contracts when the result is Yes. No-taker statistics are reported separately.

In [11]:
if selected_markets.empty or trade_summary.empty:
    market_trade_results = pd.DataFrame()
else:
    market_trade_results = selected_markets.merge(
        trade_summary, on=["ticker", "result"], how="left", validate="one_to_one",
    )
market_trade_results.head()

,occurrence_date_ny,ticker,event_ticker,is_combo,sports_series_tickers,title,result,market_volume_contracts,occurrence_datetime,occurrence_source,...,mve_collection_ticker,mve_selected_legs,total_contract,yes_contract,yes_dollar_volume,yes_trade,no_contract,no_dollar_volume,no_trade,yes_taker_pnl
0,2026-08-18,KXATPCHALLENGERMATCH-26AUG18BLAMEL-MEL,KXATPCHALLENGERMATCH-26AUG18BLAMEL,False,KXATPCHALLENGERMATCH,Will Felipe Meligeni Alves win the Blanch vs M...,yes,1131686.45,2026-08-18 23:00:00+00:00,market,...,NaN,NaN,1131686.45,802945.53,391289.8104,5130,328740.92,167365.5790,1961,-411655.7196
1,2026-08-18,KXATPCHALLENGERMATCH-26AUG18CLACHI-CHI,KXATPCHALLENGERMATCH-26AUG18CLACHI,False,KXATPCHALLENGERMATCH,Will Clement Chidekh win the Clarke vs Chidekh...,yes,1338335.92,2026-08-18 20:20:00+00:00,market,...,NaN,NaN,1338335.92,876196.79,599726.3166,6393,462139.13,113804.3624,1864,-276470.4734
2,2026-08-18,KXATPCHALLENGERMATCH-26AUG18JOHSAR-JOH,KXATPCHALLENGERMATCH-26AUG18JOHSAR,False,KXATPCHALLENGERMATCH,Will Garrett Johns win the Johns vs Saraiva Do...,yes,1467806.28,2026-08-18 19:10:00+00:00,market,...,NaN,NaN,1467806.28,1058371.16,685926.0044,6802,409435.12,140599.8291,2175,-372445.1556
3,2026-08-18,KXATPCHALLENGERMATCH-26AUG18LAJDAN-LAJ,KXATPCHALLENGERMATCH-26AUG18LAJDAN,False,KXATPCHALLENGERMATCH,Will Dusan Lajovic win the Lajovic vs Daniel: ...,no,1124026.36,2026-08-18 19:10:00+00:00,market,...,NaN,NaN,1124026.36,737994.85,340468.5919,5389,386031.51,254511.4674,1991,340468.5919
4,2026-08-18,KXATPCHALLENGERMATCH-26AUG18LLACOM-COM,KXATPCHALLENGERMATCH-26AUG18LLACOM,False,KXATPCHALLENGERMATCH,Will Francisco Comesana win the Llamas Ruiz vs...,no,1828152.27,2026-08-19 02:10:00+00:00,market,...,NaN,NaN,1828152.27,1409161.48,538271.1974,7345,418990.79,290978.9482,2287,538271.1974


## Weekly refresh note

For the first run keep `REFRESH_SINCE = None`. On a later run, set it to the previous run's `RUN_STARTED_AT` and rerun the notebook with the same `MARKETS_CSV`. In low-memory disk mode, refresh rows are written to a temporary file and automatically chunk-merged into `sports_markets.csv` by ticker, preferring refreshed rows. The existing CSV is replaced only after the pull and merge succeed.